In [ ]:
# ==============================================================================
#                      Multimodal Medical AI Chatbot for Google Colab
# ==============================================================================

print("Starting setup for Multimodal Medical AI Chatbot...")

# ✅ STEP 1: Install dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -U transformers bitsandbytes accelerate gradio openai-whisper gtts pillow nltk nest_asyncio
!apt-get -qq install ffmpeg -y

# ✅ STEP 2: Imports
import torch, os, tempfile, warnings
import nest_asyncio
import nltk
from PIL import Image
from gtts import gTTS
import whisper
import gradio as gr
from nltk import sent_tokenize
from transformers import BitsAndBytesConfig, AutoProcessor, LlavaForConditionalGeneration

nltk.download('punkt', quiet=True)
nest_asyncio.apply()

# ✅ STEP 3: Setup quantization for LLaVA
quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
model_id = "llava-hf/llava-1.5-7b-hf"

print(f"\n--- Loading LLaVA model: {model_id} ---")
processor = AutoProcessor.from_pretrained(model_id)
model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("✅ LLaVA model loaded successfully.")

# ✅ STEP 4: Load Whisper
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n--- Loading Whisper model on {DEVICE} ---")
whisper_model = whisper.load_model("base", device=DEVICE)
print("✅ Whisper model loaded.")

# ✅ STEP 5: Helper Functions

def img2txt(input_text, input_image):
    if isinstance(input_image, str):
        if not os.path.exists(input_image):
            return "Error: Image file not found."
        image = Image.open(input_image).convert("RGB")
    elif input_image is None:
        return "No image provided for analysis."
    else:
        image = input_image.convert("RGB")

    prompt = f"USER: <image>\nAct as a medical assistant. Analyze the image and respond to this prompt: {input_text or 'Describe this medical image and analyze it for potential issues.'}\nASSISTANT:"
    print(f"[LLaVA] Prompt: {prompt[:80]}...")

    inputs = processor(text=prompt, images=image, return_tensors="pt").to(model.device, dtype=model.dtype)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.7, top_p=0.9)
    text = processor.decode(output[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

    if text.endswith("</s>"):
        text = text[:-4].strip()
    if text.startswith("ASSISTANT:"):
        text = text[len("ASSISTANT:"):].strip()

    print(f"[LLaVA] Output: {text}")
    return text

def transcribe(audio_path):
    if not audio_path:
        return ""
    print(f"[Whisper] Transcribing: {audio_path}")
    try:
        result = whisper_model.transcribe(audio_path, fp16=torch.cuda.is_available())
        transcription = result["text"]
        print(f"[Whisper] Transcription: '{transcription}'")
        return transcription
    except Exception as e:
        print(f"[Whisper ERROR]: {e}")
        return f"Error transcribing audio: {e}"

def text_to_speech(text):
    if not text:
        return None
    print(f"[gTTS] Generating speech for: {text[:50]}...")
    try:
        tts = gTTS(text=text, lang='en', slow=False)
        with tempfile.NamedTemporaryFile(delete=False, suffix=".mp3", mode="wb") as fp:
            tts.write_to_fp(fp)
            return fp.name
    except Exception as e:
        print(f"[gTTS ERROR]: {e}")
        return None

# ✅ STEP 6: Gradio Interface Logic
def process_inputs(audio_path, image_path):
    speech_to_text_output = ""
    chat_output = ""
    audio_reply = None

    if audio_path:
        speech_to_text_output = transcribe(audio_path)
        if os.path.exists(audio_path):
            os.remove(audio_path)

    if image_path:
        chat_output = img2txt(speech_to_text_output, image_path)
    elif speech_to_text_output:
        chat_output = f"You said: '{speech_to_text_output}'. Please upload an image for analysis."
    else:
        chat_output = "Please provide a voice query or an image."

    if chat_output:
        audio_reply = text_to_speech(chat_output)

    return speech_to_text_output, chat_output, audio_reply

# ✅ STEP 7: Launch Gradio Interface
iface = gr.Interface(
    fn=process_inputs,
    inputs=[
        gr.Audio(sources=["microphone"], type="filepath", label="🎤 Speak your symptoms"),
        gr.Image(type="filepath", label="🩻 Upload an X-ray or medical image")
    ],
    outputs=[
        gr.Textbox(label="📝 Speech to Text"),
        gr.Textbox(label="🤖 AI Response"),
        gr.Audio(label="🔊 Voice Reply")
    ],
    title="🩺 Multimodal Medical AI Assistant",
    description="Speak or type your question and upload a medical image (e.g., X-ray). The assistant will analyze and reply with voice + text.",
)

iface.launch(inline=False, debug=True, share=True)
